# Evil twin attack

# Coding problem, 1.3.b: Detect evil-twin access points

**Skill:** Analyze Risk, Detect Attacks · **Maps to:** 1.3.B · **Language:** Python 3 (pytest)

## Problem
A wireless scanner returns a list of detected access points. Each AP is a dict with keys:
- `ssid` (str), the human-readable network name
- `bssid` (str), the hardware MAC address of the radio (normalized to uppercase)
- `security` (str), one of `"OPEN"`, `"WEP"`, `"WPA"`, `"WPA2"`, `"WPA3"`
- `signal` (int), received signal strength in dBm (negative; stronger = closer to 0)

An AP is **suspicious** (potential evil twin) if ANY of the following holds:

1. **Duplicate SSID / multiple BSSIDs**, the same SSID is being broadcast by more than one
   BSSID in the scan. Every BSSID in such a group is flagged.
2. **Security downgrade**, an SSID appears with at least two different security levels and the
   AP in question does not have the strongest security level seen for that SSID. Those weaker
   entries are flagged.

Rules combine: an AP is flagged if either condition is true for it.

## Function contract
Implement `solve(aps: list[dict]) -> list[str]` in the code editor
(see the Reference section below).

Return a **sorted, de-duplicated** list of suspicious BSSIDs.

## Input
A list of dicts, each with keys `ssid`, `bssid`, `security`, `signal`.
The list may be empty. BSSIDs are already uppercase.

## Output
A sorted list of BSSIDs (strings) that are suspicious. Empty list if none.

## Security ordering (weakest to strongest)
`OPEN` < `WEP` < `WPA` < `WPA2` < `WPA3`

## Examples
```
# Two APs share SSID "CafeGuest" — different BSSIDs -> both flagged (rule 1)
# The OPEN entry also has weaker security (rule 2)
aps = [
    {"ssid": "CafeGuest", "bssid": "AA:BB:CC:11:22:33", "security": "WPA2", "signal": -55},
    {"ssid": "CafeGuest", "bssid": "DE:AD:BE:EF:00:01", "security": "OPEN", "signal": -42},
    {"ssid": "HotelLobby","bssid": "11:22:33:44:55:66", "security": "WPA2", "signal": -65},
]
solve(aps) -> ["AA:BB:CC:11:22:33", "DE:AD:BE:EF:00:01"]

# Empty scan
solve([]) -> []

# Single SSID, single BSSID, no anomaly
solve([{"ssid": "HomeNet", "bssid": "00:11:22:33:44:55", "security": "WPA3", "signal": -50}]) -> []
```

## Constraints
- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(aps: list[dict]) -> list[str]
```

### Input (`aps`)
A list of access-point dicts. Each dict has exactly these keys:
- `ssid` (str), human-readable network name
- `bssid` (str), hardware MAC address, uppercase, colon-separated (e.g. `"AA:BB:CC:11:22:33"`)
- `security` (str), one of `"OPEN"`, `"WEP"`, `"WPA"`, `"WPA2"`, `"WPA3"`
- `signal` (int), dBm (negative integer; e.g. -55)

### Output (`result`)
A **sorted, de-duplicated** `list[str]` of suspicious BSSIDs. Empty list if no anomaly.

### Detection rules (both are OR-combined per AP)
1. **Duplicate SSID**: an SSID appears with more than one distinct BSSID in the scan.
   All BSSIDs for that SSID are flagged.
2. **Security downgrade**: an SSID appears with multiple security levels. Any AP for that
   SSID whose security level is not the maximum (strongest) for that SSID is flagged.

Security strength order (weakest to strongest):
`OPEN` < `WEP` < `WPA` < `WPA2` < `WPA3`

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- BSSIDs in input are already uppercase.
- Output is always sorted ascending and de-duplicated.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 1.3.b. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import List


def solve(aps: List[dict]) -> List[str]:
    """Return a sorted, de-duplicated list of suspicious BSSIDs from a wireless scan.

    An AP is suspicious if:
      1. Its SSID is advertised by more than one BSSID (duplicate-SSID rule), OR
      2. Its SSID appears with multiple security levels and this AP does not have
         the strongest security level seen for that SSID (security-downgrade rule).

    Security strength order (weakest -> strongest): OPEN < WEP < WPA < WPA2 < WPA3

    Args:
        aps: list of dicts with keys ssid, bssid, security, signal
    Returns:
        sorted list of suspicious BSSIDs (empty list if none)
    """
    # TODO: implement the two detection rules described above and in the Reference section below
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------

def test_duplicate_ssid_flags_all():
    """Rule 1: same SSID, two different BSSIDs -> both flagged."""
    aps = [
        {"ssid": "CafeGuest", "bssid": "AA:BB:CC:11:22:33", "security": "WPA2", "signal": -55},
        {"ssid": "CafeGuest", "bssid": "DE:AD:BE:EF:00:01", "security": "WPA2", "signal": -42},
        {"ssid": "HotelLobby", "bssid": "11:22:33:44:55:66", "security": "WPA2", "signal": -65},
    ]
    result = impl.solve(aps)
    assert result == ["AA:BB:CC:11:22:33", "DE:AD:BE:EF:00:01"]


def test_security_downgrade_flags_weaker():
    """Rule 2: same SSID with different security -> weaker entry flagged."""
    aps = [
        {"ssid": "Airport-Free", "bssid": "00:11:22:33:44:55", "security": "WPA2", "signal": -70},
        {"ssid": "Airport-Free", "bssid": "FF:EE:DD:CC:BB:AA", "security": "OPEN", "signal": -48},
        {"ssid": "SafeNet", "bssid": "AA:00:BB:11:CC:22", "security": "WPA3", "signal": -60},
    ]
    result = impl.solve(aps)
    # Both rule 1 (dup SSID) and rule 2 (OPEN < WPA2) apply; stronger BSSID also flagged by rule 1
    assert "FF:EE:DD:CC:BB:AA" in result  # weaker entry always flagged
    assert "00:11:22:33:44:55" in result  # rule 1 also flags the stronger entry
    assert "AA:00:BB:11:CC:22" not in result  # clean single BSSID WPA3


def test_clean_scan_empty_result():
    """No anomaly: each SSID has exactly one BSSID and consistent security."""
    aps = [
        {"ssid": "HomeNet", "bssid": "00:11:22:33:44:55", "security": "WPA3", "signal": -50},
        {"ssid": "OfficeNet", "bssid": "66:77:88:99:AA:BB", "security": "WPA2", "signal": -60},
    ]
    assert impl.solve(aps) == []


def test_empty_input():
    assert impl.solve([]) == []


# ---- hidden / edge cases -------------------------------------------------

def test_three_bssids_same_ssid():
    """Three BSSIDs for one SSID: all three flagged by rule 1."""
    aps = [
        {"ssid": "MallWifi", "bssid": "AA:AA:AA:AA:AA:01", "security": "WPA2", "signal": -50},
        {"ssid": "MallWifi", "bssid": "AA:AA:AA:AA:AA:02", "security": "WPA2", "signal": -55},
        {"ssid": "MallWifi", "bssid": "AA:AA:AA:AA:AA:03", "security": "WPA2", "signal": -60},
    ]
    result = impl.solve(aps)
    assert result == ["AA:AA:AA:AA:AA:01", "AA:AA:AA:AA:AA:02", "AA:AA:AA:AA:AA:03"]


def test_downgrade_only_open_flagged_when_no_dup():
    """Rule 2 alone (same SSID, two BSSIDs, different sec).
    Since two BSSIDs are present, rule 1 also fires, but let's verify both are flagged."""
    aps = [
        {"ssid": "LibraryNet", "bssid": "BB:CC:DD:EE:FF:01", "security": "WPA3", "signal": -55},
        {"ssid": "LibraryNet", "bssid": "BB:CC:DD:EE:FF:02", "security": "OPEN",  "signal": -40},
    ]
    result = impl.solve(aps)
    assert "BB:CC:DD:EE:FF:02" in result  # weaker + dup SSID
    assert "BB:CC:DD:EE:FF:01" in result  # dup SSID (rule 1)


def test_single_ap_no_flag():
    """One AP, unique SSID -> nothing flagged."""
    aps = [{"ssid": "OnlyNet", "bssid": "11:22:33:44:55:66", "security": "WPA2", "signal": -70}]
    assert impl.solve(aps) == []


def test_output_sorted_and_deduped():
    """Output must be sorted ascending and contain no duplicate BSSIDs."""
    aps = [
        {"ssid": "Net", "bssid": "CC:CC:CC:CC:CC:CC", "security": "WPA2", "signal": -50},
        {"ssid": "Net", "bssid": "AA:AA:AA:AA:AA:AA", "security": "OPEN",  "signal": -45},
        {"ssid": "Net", "bssid": "BB:BB:BB:BB:BB:BB", "security": "WPA",   "signal": -55},
    ]
    result = impl.solve(aps)
    assert result == sorted(set(result))
    assert len(result) == len(set(result))


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
